# Oxford Flower 花分类

本实训项目以Oxford Flowers花卉分类为主题，旨在通过深度学习技术实现对花卉图像的自动分类。通过本项目，学员将深入学习如何处理图像数据、构建深度学习模型、优化模型性能，并最终实现高准确率的花卉分类模型。                        



## 1. 项目背景介绍

Oxford Flowers 数据集是一个花卉集合数据集，主要用于图像分类，它分为 102 个类别共计 102 种花，其中每个类别包含 40 到 258 张图像。  
   该数据集由牛津大学工程科学系于 2008 年发布，相关论文有《Automated flower classification over a large number of classes》。
    
该数据集的特点是：
*   花卉的种类覆盖了英国常见的花卉，具有较高的实用性和代表性。
*   图像具有大的尺度、姿态和光照变化，增加了分类的难度和挑战性。
*   类别之间存在较大的内部变化和相似性，需要更精细的特征提取和判别能力。

数据文件列表

`flower_data` ---数据集目录    
`cat_to_name.json` ---一个对应花卉的品种名的字典  
`flower_data.zip` ---数据集压缩文件


## 2. 数据集分析和可视化


### 2.1 导入必要的库

In [ ]:
import numpy as np
import random
from PIL import Image
import os
import mindspore
from mindspore import nn
from mindspore.dataset import transforms, vision
from mindspore.dataset import GeneratorDataset
import matplotlib.pyplot as plt
from mindspore import ops
import pandas as pd
import argparse
import warnings
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

### 2.2 创建数据加载类

In [ ]:
class Flower_data():
    def __init__(self, source_root, train_rate=0.6, mode='train', transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        # 设置预处理
        if self.transform is None:
            self.transform = transforms.Compose([
                vision.Resize((256, 256)),
                vision.Rescale(1.0 / 255.0, 0),
                vision.RandomCrop((224, 224)),
                vision.Normalize(mean=(0.485,), std=(0.229,)),
                vision.HWC2CHW()
            ])
        kind_list = os.listdir(source_root)

        # 加载数据
        for kind in kind_list:
            images_list = os.listdir(os.path.join(source_root, kind))
            for images in images_list:
                self.images.append(os.path.join(source_root, kind, images))
                self.labels.append(int(kind) - 1)  # 因为没有0，所以-1补充

        # 随机打乱顺序
        state = np.random.get_state()
        np.random.shuffle(self.images)
        np.random.set_state(state)
        np.random.shuffle(self.labels)

        #         print(*zip(self.images, self.labels))

        # 划分训练和验证集
        assert mode in ['train', 'valid', 'test']
        if mode == 'train':
            self.images = self.images[:int(len(self.images) * train_rate)]
            self.labels = self.labels[:int(len(self.labels) * train_rate)]
        elif mode == 'valid':
            self.images = self.images[int(len(self.images) * train_rate):]
            self.labels = self.labels[int(len(self.labels) * train_rate):]


        # print(self.labels)

    def __getitem__(self, index):
        #         image_data = cv2.imread(self.images[index], cv2.IMREAD_COLOR)
        image_data = Image.open(self.images[index])
        #         image_data = cv2.resize(image_data, (224, 224))
        image_data = np.array(image_data)
        #         image_data = self.transform(image_data)
        return mindspore.Tensor(image_data,mindspore.float32), mindspore.Tensor(self.labels[index], mindspore.int32)


    def __len__(self):
        return len(self.labels)

固定随机种子

In [ ]:
def fix_seed(SEED):
    random.seed(SEED)
    np.random.seed(SEED)
    mindspore.set_seed(SEED)

In [ ]:
fix_seed(1234)

查找对应品种名的字典

In [ ]:
def find_name(json_file='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/cat_to_name.json'):
    with open(json_file, 'r') as f:
        dic = eval(''.join(f.readlines()))
        return dic


### 2.3 定义显示图像数据的函数


In [ ]:
def show_5_image():
    """
    显示数据集中的5个数据
    """
    data = Flower_data(source_root='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/flower_data/train', train_rate=0.6, mode='train')
    dataset = GeneratorDataset(source=data, column_names=["data", "label"], shuffle=False)
    dataset = dataset.map(operations=data.transform, num_parallel_workers=1)
    dic = find_name()
    plt.figure()
    for i, (data, label) in enumerate(dataset):
        if i >= 5:
            break
        plt.subplot(2, 3, i + 1)
        plt.title("label:" + dic[str(label)])
        show_image = np.transpose(np.array(data, np.float32), (1, 2, 0))
        plt.imshow(show_image)
        plt.axis("off")
    plt.show()

In [ ]:
show_5_image() 


### 2.4 数据集数量分析
横坐标表示花种类的索引，纵坐标表示该种类的训练集图像数量



In [ ]:
fix_seed(1)
flower_data = Flower_data(source_root='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/flower_data/train', train_rate=1, mode='train')

all_count = dict()
for image, label in flower_data:
    try:
        all_count[str(label)] += 1
    except KeyError:
        all_count[str(label)] = 0
# print(all_count)
kind_list = []
value_list = []
for key, value in all_count.items():
    kind_list.append(int(key))
    value_list.append(value)
all_ = zip(kind_list, value_list)
kind_list, value_list = zip(*sorted(all_, key=lambda x: x[0]))
for index in range(len(kind_list)):
    plt.bar(kind_list[index], value_list[index])
#     print(kind_list[index], value_list[index])
print("最少", np.min(value_list), '张图像')
plt.xticks(range(0, 102, 5), rotation=90)
plt.xlabel("kind_index")
plt.show()



## 3. 模型构建



### 3.1 创建模型类
这里参考 resnet，使用了残差结构，先对图片进行下采样，再在同层次的基础上使用 3x3 的卷积并加入残差，目的是防止梯度消失。  
在经过 4 个 ResidualBlock 每次再通过卷积增加通道数，同时下采样，最后通过平均池化层和全连接层进行分类。


In [ ]:
class ResidualBlock(nn.Cell):
    def __init__(self, input_c, output_c):
        super().__init__()
        self.conv1 = nn.Conv2d(input_c, output_c, kernel_size=3, stride=1)
        self.bn1 = nn.BatchNorm2d(output_c)
        self.conv2 = nn.Conv2d(output_c, output_c, kernel_size=3, stride=1)
        self.bn2 = nn.BatchNorm2d(output_c)

        self.relu = nn.ReLU()

    def construct(self, x):
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        x += residual
        return x


class Network(nn.Cell):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=(3, 3, 3, 3),
                               pad_mode='pad')
        self.bn1 = nn.BatchNorm2d(64)
        self.resi1 = ResidualBlock(64, 64)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='SAME')

        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.resi2 = ResidualBlock(128, 128)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='SAME')

        self.conv3 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.resi3 = ResidualBlock(256, 256)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='SAME')

        self.conv4 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.resi4 = ResidualBlock(512, 512)
        self.maxpool4 = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='SAME')

        self.avg = nn.AvgPool2d(kernel_size=7, stride=1)
        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()
        self.dense_relu_sequential = nn.SequentialCell(
            nn.Dense(512, 1024),
            nn.ReLU(),
            nn.Dense(1024, 512),
            nn.ReLU(),
            nn.Dense(512, 102)
        )

    def construct(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.resi1(x)
        x = self.maxpool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.resi2(x)
        x = self.maxpool2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.resi3(x)
        x = self.maxpool3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = self.resi4(x)
        x = self.maxpool4(x)

        x = self.avg(x)
        x = self.flatten(x)
        x = self.dense_relu_sequential(x)
        return x


## 4. 模型训练



### 4.1 超参数定义

In [ ]:
def get_argparse():
    parser = argparse.ArgumentParser(description=" Classification of Flower")
    parser.add_argument('--batch_size', default=8)
    parser.add_argument('--Learning_rate', default=1e-4)
    parser.add_argument('--classes', default=102)
    parser.add_argument('--max_epoch', default=20)
    parser.add_argument('--model1_name', default='UUnet')
    parser.add_argument('--device', default='cpu')
    parser.add_argument('--model_path', default=f'/home/jovyan/work/results/{now_time}/model.ckpt')
    parser.add_argument('--load_model_path', default=f'')
    parser.add_argument('--info', default=dict())
    args = parser.parse_args([])
    args.info['epoch'] = []
    args.info['train_loss'] = []
    args.info['valid_loss'] = []
    args.info['valid_Accuracy'] = []
    args.info['valid_Recall'] = []
    args.info['valid_Precision'] = []
    args.info['valid_F1_score'] = []
    return args


### 4.2 保存准确率损失等重要信息


In [ ]:
# 绘制损失函数
def draw_loss(cfg):
    plt.figure()
    plt.plot(cfg.info['epoch'], cfg.info['train_loss'], label='train_loss')
    plt.plot(cfg.info['epoch'], cfg.info['valid_loss'], label='valid_loss')
    plt.legend()
    plt.savefig(f"/home/jovyan/work/results/{now_time}/loss.png")
    plt.show()
    return


def draw_val_acc(cfg):
    plt.figure()
    plt.plot(cfg.info['epoch'], cfg.info['valid_Accuracy'], label='valid_Accuracy')
    plt.legend()
    plt.savefig(f"/home/jovyan/work/results/{now_time}/valid_Accuracy.png")
    plt.show()
    return


def save_info(cfg):
    dataframe = pd.DataFrame(cfg.info)
    dataframe.to_csv(f"/home/jovyan/work/results/{now_time}/combine_result.csv", index=False, sep=',')
    return


### 4.3 模型训练和测试


In [ ]:
def model_train(cfg, model, optimizer, loss_func, train_dataset):
    def forward_fn(data, label):
        out = model(data)
        loss = loss_func(out, label)
        return loss, out

    length = train_dataset.get_dataset_size()
    train_dataloader = train_dataset.create_tuple_iterator()
    grad_fn = ops.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)
    model.set_train()

    train_loss = []

#     train_tq = tqdm(total=length)
    for ite, (data, label) in enumerate(train_dataloader):
#         train_tq.set_description("train:")
        (loss, _), grads = grad_fn(data, label)
        loss = ops.depend(loss, optimizer(grads))
        train_loss.append(float(loss))
#         train_tq.set_postfix(loss=loss)
#         train_tq.update(1)

    train_loss = np.mean(train_loss)
    cfg.info['train_loss'].append(train_loss)
    mindspore.save_checkpoint(model, cfg.model_path)

In [ ]:
def model_eval(cfg, model, optimizer, loss_func, test_dataset):
    def forward_fn(data, label):
        out = model(data)
        loss = loss_func(out, label)
        return loss, out

    length = test_dataset.get_dataset_size()
    test_dataloader = test_dataset.create_tuple_iterator()

    model.set_train(False)

    total, test_loss, correct = 0, 0, 0
#     test_tq = tqdm(total=length)
    for ite, (data, label) in enumerate(test_dataloader):
#         test_tq.set_description("valid:")
        out = model(data)
        total += len(data)
        loss = loss_func(out, label).asnumpy()
        test_loss += loss
        correct += (out.argmax(1) == label).asnumpy().sum()

#         test_tq.set_postfix(loss=loss)
#         test_tq.update(1)

    test_loss /= length
    correct /= total

    cfg.info['valid_loss'].append(test_loss)
    cfg.info['valid_Accuracy'].append(correct)


### 4.4 开始训练


- 使用交叉熵作为损失函数
- 使用 Adam 作为优化器

下面给出训练代码  
使用 gpu 运行训练代码，具体训练代码文件见 `train.py` 

In [ ]:
warnings.filterwarnings('ignore')
now_time = time.strftime('%Y_%m_%d_%H_%M')

try:
    os.mkdir(f'/home/jovyan/work/results/{now_time}/')
except:
    pass


# 定义超参数
cfg = get_argparse()
print(cfg)

# 数据集的构建
train_data = Flower_data(source_root='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/flower_data/train', train_rate=0.6, mode='train')
train_dataset = GeneratorDataset(source=train_data, column_names=["data", "label"],
                                 shuffle=True).map(operations=train_data.transform,
                                                   num_parallel_workers=1).batch(batch_size=cfg.batch_size)

test_data = Flower_data(source_root='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/flower_data/train', train_rate=0.6, mode='valid')
test_dataset = GeneratorDataset(source=test_data, column_names=["data", "label"],
                                shuffle=False).map(operations=test_data.transform,
                                                   num_parallel_workers=1).batch(batch_size=cfg.batch_size)

# 模型，优化器，损失函数
model = Network()
loss_func = nn.CrossEntropyLoss()
optimizer = nn.Adam(model.trainable_params(), learning_rate=cfg.Learning_rate)

# 开始迭代训练
for epoch in range(cfg.max_epoch):
    cfg.info['epoch'].append(epoch + 1)
    # 可视化输出
    model_train(cfg, model, optimizer, loss_func, train_dataset)
    print('{{"metric": "epoch", "value": {}}}'.format(epoch + 1))
    model_eval(cfg, model, optimizer, loss_func, test_dataset)
    print('{{"metric": "accuracy", "value": {}}}'.format(cfg.info["valid_Accuracy"][-1]))
#     print(epoch + 1, 'valid_Accuracy ---', cfg.info['valid_Accuracy'][-1])
draw_loss(cfg)
draw_val_acc(cfg)


最终结果保存在 `results` 文件夹下，以运行的开始时间命名的文件夹中（此次训练结果在 `2025_08_04_14_307` 文件夹下）。  
同时保存模型并命名为 `model.ckpt`，保存损失图并命名为 `loss.png`, 保存准确率并命名为 `valid_Accuracy.png`。


## 5. 模型评估


- 使用准确率作为模型评估的指标

In [ ]:
now_time = time.strftime('%Y_%m_%d_%H_%M')
# 定义超参数
cfg = get_argparse()
print(cfg)

# 数据集的构建

test_data = Flower_data(source_root='/home/jovyan/work/datasets/688c8679b5074ef61818a082-momodel/flower_data/valid', train_rate=0.6, mode='test')
test_dataset = GeneratorDataset(source=test_data, column_names=["data", "label"],
                                shuffle=False).map(operations=test_data.transform,
                                                   num_parallel_workers=1).batch(batch_size=cfg.batch_size)

# 模型，优化器，损失函数
model = Network()
loss_func = nn.CrossEntropyLoss()
optimizer = nn.Adam(model.trainable_params(), learning_rate=cfg.Learning_rate)
cfg.load_model_path = '/home/jovyan/work/results/2025_08_05_13_42/model.ckpt'
param_dict = mindspore.load_checkpoint(cfg.load_model_path)
mindspore.load_param_into_net(model, param_dict)
model_eval(cfg, model, optimizer, loss_func, test_dataset)
print('valid_Accuracy ---', cfg.info['valid_Accuracy'][-1])

## 6. 总结与改进
* 总结

     本项目通过 mindspore 构建网络对 102 种花卉数据集进行分类，在运行 100 个迭代次数的条件下，在测试集上最终准确率为 47.9%。  
     从图中可以看出，在 40 次迭代后，训练损失在下降但验证损失反而上升，说明此时出现了过拟合现象，应该在网络中加入 Dropout 层使得缓解这种过拟合现象

* 改进

     可以参考 Big Transfer (BiT): General Visual Representation Learning（ECCV2020），加入注意力机制层，提升准确率。  
     但引入 transformer 会导致参数量巨增，且由于 transformer 不容易训练，要想效果好最后需要预训练模型支持。